**Goal:** Show that SHAP attributions for this model exhibit large sample-to-sample variability,
and that common summaries such as mean absolute SHAP values discard sign information that is
structurally meaningful in our kernel-based models.  This motivates the use of learned integration
kernels, which are stable across samples and preserve sign.

In [2]:
import os
import sys
import json
import shap
import torch
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from math import prod
import matplotlib.ticker as mticker
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({'toplabel.weight':'normal','leftlabel.weight':'normal','fontsize':14,'figure.dpi':100})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR   = CONFIGS['filepaths']['splits']
WEIGHTSDIR  = CONFIGS['filepaths']['weights']
MODELSDIR   = CONFIGS['filepaths']['models']
MODELS      = CONFIGS['models']
FIELDVARS   = CONFIGS['variables']['field']  
LOCALVARS   = CONFIGS['variables']['local']  
TARGETVAR   = CONFIGS['variables']['target']  
LATRANGE    = CONFIGS['domain']['latrange']
LONRANGE    = CONFIGS['domain']['lonrange']
SEEDS       = CONFIGS['training']['seeds']    
SPLIT       = 'test'
MODELNAME   = 'baseline_nonlocal_vertical'
MODELCONFIG = next(m for m in MODELS if m['name']==MODELNAME)
PATCHCONFIG = MODELCONFIG['patch']   
USELOCAL    = MODELCONFIG['uselocal']
MAXRADIUS   = max(m['patch']['radius']  for m in MODELS)
MAXTIMELAG  = max(m['patch']['timelag'] for m in MODELS)
NFIELDVARS  = len(FIELDVARS)+1
NLOCALVARS  = len(LOCALVARS)     

print(f'                           Model name = {MODELNAME}')
print(f'                          PatchConfig = {PATCHCONFIG}')
print(f'                            Use local = {USELOCAL}')
print(f'Number of field vars (including mask) = {NFIELDVARS}')
print(f'                 Number of local vars = {NLOCALVARS}')

                           Model name = baseline_nonlocal_vertical
                          PatchConfig = {'radius': 0, 'levmode': 'column', 'timelag': 0}
                            Use local = True
Number of field vars (including mask) = 4
                 Number of local vars = 3


In [4]:
from scripts.data.classes import PatchDataLoader

splitdata = PatchDataLoader.prepare(
    [SPLIT],
    FIELDVARS,
    LOCALVARS,
    TARGETVAR,
    SPLITSDIR)

result = PatchDataLoader.dataloaders(
    splitdata,
    PATCHCONFIG,
    USELOCAL,
    LATRANGE,
    LONRANGE,
    batchsize=500,
    workers=0,
    device='cpu',
    maxradius=MAXRADIUS,
    maxtimelag=MAXTIMELAG)

dataset    = result['datasets'][SPLIT]
loader     = result['loaders'][SPLIT]
patchshape = dataset.shape()
nlevs      = patchshape[2]

with xr.open_dataset(f'{SPLITSDIR}/{SPLIT}.h5',engine='h5netcdf') as ds:
    lev = ds['lev'].values.astype(np.float32)

print(f'       Patch shape = {patchshape}')
print(f'  Number of levels = {nlevs}')
print(f'Total test samples = {len(dataset):,}')

       Patch shape = (1, 1, 16, 1)
  Number of levels = 16
Total test samples = 3,646,518


In [5]:
NBACKGROUND = 200 # For background reference
NTEST       = 500 # For explanation
NTOTAL      = NBACKGROUND+NTEST # Total sample subset for SHAP

chunks = []
for batch in loader:
    fieldpatch  = batch['fieldpatch']   
    localvalues = batch['localvalues']  
    X  = torch.cat([fieldpatch.flatten(1),localvalues],dim=1).detach()
    chunks.append(X)
    if sum(c.shape[0] for c in chunks)>=NTOTAL:
        break

Xpool = torch.cat(chunks,dim=0)[:NTOTAL]
rng   = np.random.default_rng(SEEDS[0])
perm  = rng.permutation(NTOTAL)
Xbackground,Xtest  = Xpool[perm[:NBACKGROUND]].float(),Xpool[perm[NBACKGROUND:]].float()

print(f'  Background tensor = {Xbackground.shape}')
print(f'        Test tensor = {Xtest.shape}')
print(f'Features per sample = {Xbackground.shape[1]}  ( = {NFIELDVARS}×{nlevs} + {NLOCALVARS})')

  Background tensor = torch.Size([200, 67])
        Test tensor = torch.Size([500, 67])
Features per sample = 67  ( = 4×16 + 3)


In [6]:
from scripts.models.classes import ModelFactory

SEED = SEEDS[0]

model = ModelFactory.build(
    MODELNAME,
    MODELCONFIG,
    patchshape,
    NFIELDVARS,
    NLOCALVARS)

checkpointpath = os.path.join(MODELSDIR,f'{MODELNAME}_{SEED}.pth')
state = torch.load(checkpointpath,map_location='cpu')
model.load_state_dict(state)
model.eval()

print(f'Loaded = {checkpointpath}')
print(f'Params = {sum(p.numel() for p in model.parameters()):,}')

Loaded = /global/cfs/cdirs/m4334/sferrett/monsoon-kernels/models/baseline_nonlocal_vertical_42.pth
Params = 60,673


In [7]:
class FlatBaselineNN(torch.nn.Module):
    '''
    Thin wrapper around BaselineNN that accepts a concatenated flat input [fieldpatch.flatten(1) | localvalues] 
    and returns scalar predictions. The split point between fieldpatch and localvalues is 
    `fieldpatchsize` = NFIELDVARS*prod(patchshape).
    '''
    def __init__(self, model,nfieldvars,patchshape,nlocalvars):
        super().__init__()
        self.model           = model
        self.fieldpatchshape = (nfieldvars,*patchshape)   
        self.fieldpatchsize  = prod(self.fieldpatchshape)         
        self.nlocalvars      = nlocalvars
    def forward(self, X):
        fieldpatch  = X[:,:self.fieldpatchsize].reshape(-1,*self.fieldpatchshape)
        localvalues = X[:,self.fieldpatchsize:]
        return self.model(fieldpatch,localvalues)  

wrapper = FlatBaselineNN(model,NFIELDVARS,patchshape,NLOCALVARS)
wrapper.eval()

with torch.no_grad():
    ycheck = wrapper(Xtest[:4])
print(f'Wrapper output shape for 4 samples: {ycheck.shape}')

Wrapper output shape for 4 samples: torch.Size([4])


In [8]:
explainer = shap.DeepExplainer(wrapper,Xbackground)
shapraw   = explainer.shap_values(Xtest)   

if isinstance(shapraw,list):
    shapraw = shapraw[0]
if shapraw.ndim==3:
    shapraw = shapraw[:,:,0]
shapraw = np.array(shapraw,dtype=np.float32) 

shapfields = shapraw[:,:3*nlevs].reshape(NTEST,3,nlevs) 
shaplocal  = shapraw[:,4*nlevs:]                             

print(f'SHAP array shape = {shapraw.shape}')
print(f'Field SHAP shape = {shapfields.shape} [samples, vars, levels]')
print(f'Local SHAP shape = {shaplocal.shape} [samples, localvars]')

IndexError: tuple index out of range

## Plot 1 — Mean SHAP values and sample-to-sample variability

Each panel shows the mean SHAP attribution across 500 test samples (solid line)
together with a ±1 σ shading and a selection of 20 individual sample profiles (thin lines).
The wide spread illustrates that the network does not attribute contributions to vertical levels
in a consistent way: the same level may receive a large positive attribution for one sample
and a large negative one for another.

In [10]:
FIELDLABELS  = ['RH',r'$\mathit{\theta_e}$',r'$\mathit{\theta_e^*}$']
NINDIVIDUAL  = 20

shapmean = shapfields.mean(axis=0)   
shapstd  = shapfields.std(axis=0)   

rngplot   = np.random.default_rng(0)
sampleidx = rngplot.choice(NTEST,NINDIVIDUAL,replace=False)

fig,axs = pplt.subplots(nrows=1,ncols=3,sharey=True,width=8,height=5,wspace=1)
axs.format(collabels=FIELDLABELS,grid=False,xlabel='SHAP Value',
           ylabel='Pressure (hPa)',ylim=(500,1000),yticks=100,yminorticks='none')
for col in range(3):
    ax   = axs[col]
    mean = shapmean[col] 
    std  = shapstd[col]   
    for i in sampleidx:
        ax.plot(shapfields[i,col],lev,color='blue6',
                alpha=0.12,linewidth=0.7,zorder=1)
    ax.fill_betweenx(lev,mean-std,mean+std,color='blue6',alpha=0.30,label=r'$\pm 1\sigma$',zorder=2)
    ax.plot(mean,lev,color='blue6',linewidth=1.8,label='Mean',zorder=3)
    ax.axvline(0,color='k',linewidth=0.5,zorder=0)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4,prune='both'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
axs.format(yreverse=True)
axs[0].legend(loc='ur',ncols=1,frame=False,prop={'size':11})
pplt.show()
# fig.save('../figs/shap_mean_variability.png',dpi=300)

## SHAP values for local surface variables

For completeness, the cell below shows the distribution of SHAP attributions for the three
local surface inputs (land fraction, latent heat flux, sensible heat flux).
These are scalar inputs (not vertical profiles), so variability is displayed as violin plots.

In [ ]:
LOCALLABELS = ['Land\nfraction\n(lf)','Latent\nheat flux\n(lhf)','Sensible\nheat flux\n(shf)']

fig,ax = pplt.subplots(width=5,height=3.5)
ax.format(grid=False,xticklabels=LOCALLABELS,ylabel='SHAP Value')

pos   = np.arange(NLOCALVARS)
parts = ax.violinplot([shaplocal[:,i] for i in range(NLOCALVARS)],positions=pos,widths=0.5,showmedians=True)

for pc in parts['bodies']:
    pc.set_facecolor('blue6')
    pc.set_alpha(0.5)
for part in ('cbars','cmins','cmaxes','cmedians'):
    parts[part].set_color('blue6')

ax.axhline(0,color='k',linewidth=0.5)
ax.set_xticks(pos)
pplt.show()
# fig.save('../figs/shap_local_vars.png',dpi=300)

## Summary

The plots above illustrate three key limitations of SHAP-based interpretability for this baseline model:

1. **High sample-to-sample variability** (Plot 1): Individual SHAP profiles scatter widely around
   the mean, with standard deviations of comparable or larger magnitude than the mean itself.
   This reflects the fact that the MLP mixes vertical levels and variables through successive
   nonlinear layers; the attribution assigned to a given level depends heavily on the values
   of all other inputs in a given sample.

2. **Loss of sign information** (Plot 2, top row): Summarising attributions by their absolute
   values — the standard feature-importance view — discards the sign of contributions,
   making it impossible to say whether a level's anomaly *increases* or *decreases* the
   predicted precipitation for a typical sample.

3. **Contrast with kernel weights** (Plot 2, bottom row): The learned nonparametric kernel
   weights are stable across the three training seeds, preserve sign, and have a direct
   mathematical relationship to the prediction: the kernel-integrated feature *is* the
   scalar quantity passed to the nonlinear mapping.  This makes the kernel a principled,
   stable, and interpretable summary of vertical structure that SHAP attributions
   on the baseline model cannot provide.